# Physics-Corroborated Anomaly Detection (PCAD)

Existing anomaly detection for urban air quality relies on 
statistical features only [Liu et al. 2008, Hariri et al. 2019]. 
These methods cannot distinguish:
- (A) Real atmospheric pollution events
- (B) Sensor faults or calibration drift

**Proposed: PCAD** — integrating Gaussian Plume dispersion 
physics as a feature to score anomaly confidence.

Key insight: An anomaly at a station is *corroborated* if the 
Gaussian Plume model also predicts elevated concentration at 
that station given current wind. If the plume predicts clean 
air but the sensor flags an anomaly, it is likely a sensor fault.

This is novel: no existing paper uses dispersion physics as 
a feature in unsupervised anomaly detection for urban AQ.

References:
- Liu et al. (2008): Isolation Forest. ICDM.
- Hariri et al. (2019): Extended Isolation Forest. IEEE TKDE.
- Rollo et al. (2023): AIrSense ensemble detector. (ensemble 
  voting approach — this paper extends it with physics)

In [ ]:
import os, sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
conda_prefix = r"C:\Users\harsh\anaconda3\envs\raphael-env"
lib_bin = os.path.join(conda_prefix, "Library", "bin")
if os.path.exists(lib_bin):
    if lib_bin not in os.environ["PATH"]:
        os.environ["PATH"] = lib_bin + os.pathsep + os.environ["PATH"]
    if sys.platform == 'win32' and hasattr(os, 'add_dll_directory'):
        try: os.add_dll_directory(lib_bin)
        except: pass
import sqlite3
sys.path.insert(0, os.path.abspath('..'))  # backend root
DB_PATH = os.path.abspath('../data/raphael.db')
if not os.path.exists(DB_PATH):
    DB_PATH = os.path.abspath('data/raphael.db')
assert os.path.exists(DB_PATH), f"DB not found: {DB_PATH}"
conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH}")

PUNE_REGION_ID = conn.execute(
    "SELECT id FROM regions WHERE name='Pune Metropolitan Region'"
).fetchone()[0]
print(f"Pune region ID: {PUNE_REGION_ID}")


In [ ]:
import os, sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
conda_prefix = r"C:\Users\harsh\anaconda3\envs\raphael-env"
lib_bin = os.path.join(conda_prefix, "Library", "bin")
if os.path.exists(lib_bin):
    if lib_bin not in os.environ["PATH"]:
        os.environ["PATH"] = lib_bin + os.pathsep + os.environ["PATH"]
    if sys.platform == 'win32' and hasattr(os, 'add_dll_directory'):
        try: os.add_dll_directory(lib_bin)
        except: pass

import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'text.color':       '#c9d1d9',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#c9d1d9',
    'ytick.color':      '#c9d1d9',
    'axes.edgecolor':   '#30363d',
    'grid.color':       '#21262d',
    'savefig.facecolor':'#0d1117',
})
ACCENT = '#58a6ff'
WARN   = '#d29922'
DANGER = '#f85149'
OK     = '#3fb950'
MUTED  = '#8b949e'

ZONES = {
    'Hadapsar Industrial':  (18.5018, 73.9320),
    'Pune NE Quadrant':     (18.5632, 73.9401),
    'Kothrud Residential':  (18.5074, 73.8077),
    'Katraj Hills':         (18.4524, 73.8567),
    'Shivajinagar':         (18.5308, 73.8474),
    'Aundh':                (18.5590, 73.8080),
}


## Section 2 — Load & Align AQ + Weather Data

In [ ]:
import pandas as pd
import numpy as np
import json

# Load AQ
df_aq = pd.read_sql_query("""
    SELECT id, station_id, station_name, observed_at, value
    FROM raw_observations
    WHERE region_id = :region_id
      AND layer_type = 'aq'
      AND value > 0 AND value < 500
    ORDER BY observed_at
""", conn, params={"region_id": PUNE_REGION_ID})
df_aq['observed_at'] = pd.to_datetime(df_aq['observed_at'])

# Load Weather
df_w = pd.read_sql_query("""
    SELECT observed_at, station_name, value, raw_payload
    FROM raw_observations
    WHERE region_id = :region_id
      AND layer_type = 'weather'
      AND station_name IN ('wind_speed_10m', 'wind_direction_10m')
    ORDER BY observed_at
""", conn, params={"region_id": PUNE_REGION_ID})
df_w['observed_at'] = pd.to_datetime(df_w['observed_at'])

print("Sample weather payload raw structure:")
for i in range(min(3, len(df_w))):
    print(df_w['raw_payload'].iloc[i])

# Pivot Weather
df_w = df_w.drop_duplicates(subset=['observed_at', 'station_name'])
df_w_piv = df_w.pivot(index='observed_at', columns='station_name', values='value').reset_index()
df_w_piv = df_w_piv.rename(columns={
    'wind_speed_10m': 'wind_speed',
    'wind_direction_10m': 'wind_dir'
})
if 'wind_speed' not in df_w_piv: df_w_piv['wind_speed'] = 3.0
if 'wind_dir' not in df_w_piv: df_w_piv['wind_dir'] = 270.0

# Align AQ and Weather (nearest within 1 hour)
df_aq = df_aq.sort_values('observed_at')
df_w_piv = df_w_piv.sort_values('observed_at')

df_merged = pd.merge_asof(df_aq, df_w_piv, on='observed_at', direction='nearest', tolerance=pd.Timedelta('1 hour'))

# Fill missing weather with defaults
n_matched = df_merged['wind_speed'].notna().sum()
pct_matched = (n_matched / len(df_merged)) * 100
print(f"\nAQ matched with weather within 1h: {n_matched}/{len(df_merged)} ({pct_matched:.2f}%)")

df_merged['wind_speed'] = df_merged['wind_speed'].fillna(3.0)
df_merged['wind_dir'] = df_merged['wind_dir'].fillna(270.0)


## Section 3 — Gaussian Plume Feature Computation

In [ ]:
from ml.plume import _pg_class_from_wind, _sigma_y, _sigma_z, centre_line_concentration

STATION_COORDS = {
    'Savitribai Phule Pune University': (18.5308, 73.8473),
    'Hadapsar': (18.4983, 73.9258),
    'Katraj Dairy': (18.4500, 73.8650),
}

def compute_plume_concentration(aq_time, aq_station, wind_speed, wind_dir):
    hour = pd.Timestamp(aq_time).hour
    solar = 'moderate' if 6 <= hour <= 18 else 'night'
    stability = _pg_class_from_wind(max(float(wind_speed), 0.5), solar == 'moderate')
    
    st_lat, st_lon = STATION_COORDS.get(aq_station, (18.5308, 73.8473))
    max_conc = 0.0
    
    for zone_name, (z_lat, z_lon) in ZONES.items():
        import math
        R = 6371
        dlat = math.radians(st_lat - z_lat)
        dlon = math.radians(st_lon - z_lon)
        a = (math.sin(dlat/2)**2 +
             math.cos(math.radians(z_lat)) * math.cos(math.radians(st_lat)) *
             math.sin(dlon/2)**2)
        dist_km = R * 2 * math.asin(math.sqrt(a))
        
        if dist_km < 0.1:
            continue
            
        Q = 1000.0  # Proxy
        u = max(float(wind_speed), 0.5)
        dist_m = dist_km * 1000.0
        
        conc = centre_line_concentration(Q, u, dist_m, 5.0, stability)
        max_conc = max(max_conc, conc)
        
    return max_conc, stability

plume_concs = []
stability_classes = []

for idx, row in df_merged.iterrows():
    if idx % 500 == 0:
        print(f"Processing row {idx}/{len(df_merged)}...")
    conc, stab = compute_plume_concentration(row['observed_at'], row['station_name'], row['wind_speed'], row['wind_dir'])
    plume_concs.append(conc)
    stability_classes.append(stab)

df_merged['plume_predicted_conc'] = plume_concs
df_merged['stability_class'] = stability_classes
df_merged['plume_corroborated'] = df_merged['plume_predicted_conc'] > 10.0

print("\nStability Classes Distribution:")
print(df_merged['stability_class'].value_counts())
print(f"Plume corroborated readings (>10.0): {df_merged['plume_corroborated'].sum()} ({df_merged['plume_corroborated'].mean()*100:.2f}%)")


## Section 4 — PCAD Model Training & Comparison

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
import pickle

# Feature engineer rolling features same as Notebook 1
df_merged['hour_of_day'] = df_merged['observed_at'].dt.hour
df_merged['day_of_week'] = df_merged['observed_at'].dt.dayofweek
df_merged['is_weekend'] = df_merged['day_of_week'].isin([5, 6]).astype(int)
df_merged['rolling_mean_3h'] = df_merged.groupby('station_id')['value'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df_merged['rolling_std_3h'] = df_merged.groupby('station_id')['value'].transform(lambda x: x.rolling(3, min_periods=1).std().fillna(0.0))
df_merged['z_score'] = df_merged.groupby('station_id')['value'].transform(lambda x: (x - x.mean()) / (x.std() if x.std() > 0 else 1.0))
df_merged['delta_1h'] = df_merged.groupby('station_id')['value'].transform(lambda x: x.diff().fillna(0.0))

le = LabelEncoder()
df_merged['station_id_encoded'] = le.fit_transform(df_merged['station_id'])
le_stab = LabelEncoder()
df_merged['stability_encoded'] = le_stab.fit_transform(df_merged['stability_class'])

df_merged = df_merged.dropna().reset_index(drop=True)

split_idx = int(len(df_merged) * 0.8)
train_df = df_merged.iloc[:split_idx]
test_df = df_merged.iloc[split_idx:].copy()

feature_cols_A = ['value', 'hour_of_day', 'day_of_week', 'is_weekend', 'rolling_mean_3h', 'rolling_std_3h', 'z_score', 'delta_1h', 'station_id_encoded']
feature_cols_B = feature_cols_A + ['plume_predicted_conc', 'stability_encoded']

X_train_A = train_df[feature_cols_A]
X_test_A = test_df[feature_cols_A]
X_train_B = train_df[feature_cols_B]
X_test_B = test_df[feature_cols_B]

RECOMMENDED_CONTAM = 0.05  # Standard default if not loaded
try:
    with open('notebooks/models/if_tuned.pkl', 'rb') as f:
        if_tuned_model = pickle.load(f)
        RECOMMENDED_CONTAM = if_tuned_model.contamination
except:
    pass

if_A = IsolationForest(n_estimators=200, contamination=RECOMMENDED_CONTAM, random_state=42)
if_A.fit(X_train_A)
labels_A = if_A.predict(X_test_A)

if_B = IsolationForest(n_estimators=200, contamination=RECOMMENDED_CONTAM, random_state=42)
if_B.fit(X_train_B)
labels_B = if_B.predict(X_test_B)

# PCAD Confidence Category
corroborated = test_df['plume_corroborated'].values
pcad_confidence = np.where(
    labels_B == -1,
    np.where(corroborated, 'HIGH', 'MEDIUM'),
    'NORMAL'
)
test_df['pcad_confidence'] = pcad_confidence

print("PCAD confidence breakdown:")
print(test_df['pcad_confidence'].value_counts())

def jaccard(a, b):
    set_a = set(np.where(a == -1)[0])
    set_b = set(np.where(b == -1)[0])
    if not set_a and not set_b: return 1.0
    return len(set_a & set_b) / len(set_a | set_b)

test_df['baseline_z'] = (np.abs(test_df['z_score']) > 3).astype(int)
z_agr_A = (np.where(labels_A == -1, 1, 0) == test_df['baseline_z']).mean() * 100
z_agr_B = (np.where(labels_B == -1, 1, 0) == test_df['baseline_z']).mean() * 100

print("\n=== Table Comparison ===")
comp_data = [
    {"Metric": "N anomalies", "IF-only (A)": f"{(labels_A == -1).sum()}", "PCAD (B)": f"{(labels_B == -1).sum()}"},
    {"Metric": "High confidence", "IF-only (A)": "N/A", "PCAD (B)": f"{(pcad_confidence == 'HIGH').sum()}"},
    {"Metric": "Medium confidence", "IF-only (A)": "N/A", "PCAD (B)": f"{(pcad_confidence == 'MEDIUM').sum()}"},
    {"Metric": "Z-score agreement", "IF-only (A)": f"{z_agr_A:.2f}%", "PCAD (B)": f"{z_agr_B:.2f}%"},
    {"Metric": "Jaccard A vs B", "IF-only (A)": f"{jaccard(labels_A, labels_B):.3f}", "PCAD (B)": "1.000"}
]
comp_df = pd.DataFrame(comp_data)
print(comp_df.to_markdown(index=False))

# Scatter plot
plt.figure(figsize=(10, 6))
colors = {'HIGH': DANGER, 'MEDIUM': WARN, 'NORMAL': OK}
for cat in ['NORMAL', 'MEDIUM', 'HIGH']:
    sub = test_df[test_df['pcad_confidence'] == cat]
    plt.scatter(sub['value'], sub['plume_predicted_conc'], color=colors[cat], label=cat, alpha=0.7)
plt.title('PCAD Anomaly Confidence Categories', color='#c9d1d9')
plt.xlabel('PM2.5 Sensor Reading (μg/m³)')
plt.ylabel('Gaussian Plume Concentration (μg/m³)')
plt.legend()
plt.grid(True)
plt.savefig('outputs/02a_pcad_scatter.png')
plt.show()

# Timeline
plt.figure(figsize=(12, 6))
plt.plot(test_df['observed_at'], test_df['value'], color=MUTED, alpha=0.5, label='PM2.5')
for cat in ['MEDIUM', 'HIGH']:
    sub = test_df[test_df['pcad_confidence'] == cat]
    plt.scatter(sub['observed_at'], sub['value'], color=colors[cat], label=f'{cat} Anomaly', zorder=5)
plt.title('Anomaly Timeline with Physics Corroboration', color='#c9d1d9')
plt.xlabel('Timestamp')
plt.ylabel('PM2.5 (μg/m³)')
plt.legend()
plt.grid(True)
plt.savefig('outputs/02b_pcad_timeline.png')
plt.show()


## Section 5 — SHAP Explanations

In [ ]:
import shap

try:
    explainer = shap.TreeExplainer(if_B)
    shap_values = explainer.shap_values(X_test_B)
    shap_method = "TreeExplainer"
except Exception as e:
    background = shap.sample(X_train_B, min(100, len(X_train_B)))
    explainer = shap.explainers.Permutation(if_B.decision_function, background)
    shap_values = explainer(X_test_B).values
    shap_method = "PermutationExplainer"

print(f"SHAP method: {shap_method}")

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({
    'feature': feature_cols_B,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

print(shap_df.to_markdown(index=False))

rank = np.where(shap_df['feature'] == 'plume_predicted_conc')[0][0] + 1
print(f"\nPhysics feature SHAP rank: #{rank}")
if rank <= 5:
    print("Physics feature is informative")
else:
    print("Physics feature is not strongly informative at current data scale. Honest negative result.")

# Side by side SHAP plot
plt.figure(figsize=(10, 6))
plt.barh(shap_df['feature'][::-1], shap_df['mean_abs_shap'][::-1], color=ACCENT)
plt.title('SHAP Feature Importance with Physics (Model B)', color='#c9d1d9')
plt.grid(axis='x')
plt.savefig('outputs/02c_shap_comparison.png')
plt.show()


## Section 6 — Bootstrap Validation

In [ ]:
n_boot = 500
agreements = []
rng = np.random.default_rng(42)
z_labels = test_df['baseline_z'].values

for _ in range(n_boot):
    idx = rng.choice(len(X_test_B), len(X_test_B), replace=True)
    z_sub = z_labels[idx] == 1
    preds_sub = if_B.predict(X_test_B.iloc[idx]) == -1
    agreements.append((preds_sub == z_sub).mean() * 100)
    
ci_low = np.percentile(agreements, 2.5)
ci_high = np.percentile(agreements, 97.5)
mean_agr = np.mean(agreements)

print(f"IF-zscore bootstrap agreement: {mean_agr:.3f}%, 95% CI [{ci_low:.3f}%, {ci_high:.3f}%]")


## Section 7 — Paper Summary

In [ ]:
print("=== Table 2: PCAD Innovation Summary ===")
print(comp_df.to_markdown(index=False))
print(f"\nBootstrap Mean Agreement: {mean_agr:.2f}% (95% CI: {ci_low:.2f}% - {ci_high:.2f}%)")
print("Note: This evaluation uses z-score agreement as proxy ground truth. True precision/recall requires labeled anomaly events.")
conn.close()
